# Deploy Fine-tuned LLM to SageMaker

This notebook shows how to deploy a LoRA fine-tuned model to AWS SageMaker for inference.

## Overview
1. Merge LoRA adapters with base model
2. Test locally
3. Package and upload to S3
4. Deploy to SageMaker endpoint
5. Run inference


## Setup

Import required libraries and configure paths.


In [1]:
import os
import torch
from peft import PeftModel
from paths import OUTPUTS_DIR
from dotenv import load_dotenv
from utils.config_utils import load_config
from sagemaker.huggingface import HuggingFaceModel, HuggingFacePredictor
from transformers import AutoModelForCausalLM, AutoTokenizer


adapters_dir = os.path.join(OUTPUTS_DIR, "lora_samsum", 'lora_adapters')
merged_model_dir = os.path.join(OUTPUTS_DIR, "lora_samsum", 'merged_model')
cfg = load_config()

d:\Programing\LLMs\rt-llm-eng-cert\rt-llm-eng-cert-week6\llm-eng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\moham\AppData\Local\sagemaker\sagemaker\config.yaml


## Step 1: Merge LoRA Adapters with Base Model

SageMaker's default HuggingFace inference toolkit doesn't support loading LoRA adapters on the fly. 
We need to merge the adapters into the base model first.


In [2]:
# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Load LoRA adapters (download from S3 first or use local path)
model = PeftModel.from_pretrained(base_model, adapters_dir)

# Merge adapters into base model
merged_model = model.merge_and_unload()
# Save merged model
merged_model.save_pretrained(merged_model_dir)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
tokenizer.save_pretrained(merged_model_dir)


`torch_dtype` is deprecated! Use `dtype` instead!
d:\Programing\LLMs\rt-llm-eng-cert\rt-llm-eng-cert-week6\llm-eng\Lib\site-packages\peft\config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


('d:\\Programing\\LLMs\\rt-llm-eng-cert\\rt-llm-eng-cert-week6\\data\\outputs\\lora_samsum\\merged_model\\tokenizer_config.json',
 'd:\\Programing\\LLMs\\rt-llm-eng-cert\\rt-llm-eng-cert-week6\\data\\outputs\\lora_samsum\\merged_model\\special_tokens_map.json',
 'd:\\Programing\\LLMs\\rt-llm-eng-cert\\rt-llm-eng-cert-week6\\data\\outputs\\lora_samsum\\merged_model\\chat_template.jinja',
 'd:\\Programing\\LLMs\\rt-llm-eng-cert\\rt-llm-eng-cert-week6\\data\\outputs\\lora_samsum\\merged_model\\tokenizer.json')

## Step 2: Test Model Locally

Before deploying to SageMaker, test the merged model works correctly on a sample.


In [3]:
from utils.data_utils import load_and_prepare_dataset, build_messages_for_sample

train, val, test = load_and_prepare_dataset(cfg)

Loading dataset from local cache: d:\Programing\LLMs\rt-llm-eng-cert\rt-llm-eng-cert-week6\data\datasets\knkarthick_samsum
📊 Loaded 14731 train / 200 val / 200 test samples (from full cache).


In [4]:
train[0]

{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

In [5]:
messages = build_messages_for_sample(train[0], cfg["task_instruction"])

text_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

In [6]:
text_prompt

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 21 Jul 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nYou are a helpful assistant who writes concise, factual summaries of conversations. Summarize the following conversation into a single sentence.\n\n\n## Dialogue:\nAmanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)\n## Summary:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [7]:
from transformers import pipeline
pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer)
pipe(text_prompt, return_full_text=False)

Device set to use cpu


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': 'Amanda baked cookies and will bring them to Jerry tomorrow.'}]

## Step 3: Package and Upload to S3

SageMaker needs the model in a tar.gz file stored in S3.


In [8]:
import tarfile
import boto3

cfg = load_config()
bucket = cfg["bucket"]
s3_key = f"{cfg['output_path']}/merged_model/model.tar.gz"


In [9]:
# Create tar.gz
tar_path = os.path.join(OUTPUTS_DIR, "lora_samsum", "model.tar.gz")
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(merged_model_dir, arcname=".")

In [10]:
# Upload to S3
s3 = boto3.client("s3")
s3.upload_file(tar_path, bucket, s3_key)

print(f"Uploaded to s3://{bucket}/{s3_key}")

Uploaded to s3://sagemaker-llmeng-training/llama3-2-1b-samsum-finetune/merged_model/model.tar.gz


## Step 4: Deploy to SageMaker Endpoint

Create a HuggingFace model and deploy it to a real-time endpoint. This takes ~5-10 minutes.


In [40]:
import boto3
load_dotenv()
region = os.getenv("REGION")
# Clean up the broken endpoint
sm_client = boto3.client("sagemaker", region_name=region)
sm_client.delete_endpoint(EndpointName="llama-endpoint")
sm_client.delete_endpoint_config(EndpointConfigName="llama-endpoint")

{'ResponseMetadata': {'RequestId': '31e4272e-10c6-404e-b36a-a884fc4b1a74',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '31e4272e-10c6-404e-b36a-a884fc4b1a74',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Tue, 21 Jul 2026 18:05:21 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

In [41]:
import sagemaker
load_dotenv()

role = os.getenv("SAGEMAKER_EXECUTION_ROLE_ARN")
region = os.getenv("REGION")
bucket = cfg["bucket"]

# Create a session in the same region as your S3 bucket
boto_session = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

model = HuggingFaceModel(
    model_data=f"s3://{bucket}/{s3_key}",
    role=role,
    transformers_version="4.51",
    pytorch_version="2.6",
    py_version="py312",
    sagemaker_session=sagemaker_session,  # <-- pin to us-east-1
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",  # ml.g5.xlarge
    endpoint_name="llama-endpoint-v1"
)


------!

## Step 5: Run Inference

Connect to the deployed endpoint and test it with a sample prompt.


In [42]:
text_prompt

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 21 Jul 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nYou are a helpful assistant who writes concise, factual summaries of conversations. Summarize the following conversation into a single sentence.\n\n\n## Dialogue:\nAntonio: The games tomorrow\nIsiah: I aint goin\nAntonio: What whyyy\nIsiah: Melanies sick\nAntonio: Man you promised, Rachel could stay with her\nIsiah: She spends all the time at home I dont wanna leave her with this, there are two other kids, Ant\nAntonio: Ehhh I knowwww\nIsiah: Sorry\nAntonio: Which of the guys would use your ticket, how do ya think\nIsiah: Idk theyre all seem to be at work you need to ask\nAntonio: I want to go there so baaad\nIsiah: Just go alone\nAntonio: Its not that fun, I did it once, not cool\nIsiah: Your problem bro, I have a fuckin hell here\nAntonio: That bad?\nIsiah: Mel cryin all the time, Tim and Joe are tired 

In [44]:
predictor = HuggingFacePredictor(
    endpoint_name="llama-endpoint-v1",
    sagemaker_session=sagemaker_session,  # <-- the us-east-1 session from the deploy cell
)

predictor.predict({"inputs": text_prompt, "parameters": {
    "return_full_text": False, "do_sample": False
}})


[{'generated_text': "Isiah is going to go to the games tomorrow. He's sick and wants to go with Rachel"}]

In [45]:
# if you'd like to delete the end point
predictor.delete_endpoint()

In [ ]:
from tqdm import tqdm

results = []

for sample in tqdm(val[:10]):
    messages = build_messages_for_sample(sample, cfg["task_instruction"])

    text_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    out = predictor.predict({
        "inputs": text_prompt,
        "parameters": {"return_full_text": False, "do_sample": False}
    })
    results.append(out)


In [ ]:
results[0]

In [53]:
import evaluate
rouge = evaluate.load("rouge")

predictions = [result[0]["generated_text"] for result in results]
references = [sample["summary"] for sample in val]

rouge.compute(predictions=predictions, references=references)




{'rouge1': np.float64(0.45472632604924645),
 'rouge2': np.float64(0.22621493392056607),
 'rougeL': np.float64(0.38015757712025816),
 'rougeLsum': np.float64(0.3804240535060933)}